# Analise estatica do backend - GTA V / Steam Reviews

Este notebook gera evidencias visuais e matematicas para validar a classificacao por grafo tripartido implementada no backend.

O foco e demonstrar:

- balanceamento das reviews classificadas por categoria;
- palavras mais pesadas por TF-IDF em cada categoria;
- convergencia numerica do Label Propagation usando `max_delta` por iteracao.

Por enquanto, a analise usa os dados mockados do backend. Quando o dataset real de GTA V estiver integrado, basta trocar a origem de `documents`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

ROOT

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

from backend.graph.builder import build_tripartite_graph, mock_documents, mock_seed_groups
from backend.preprocessing.tf_idf import calculate_tf_idf
from backend.propagation.label_propagation import classify_reviews, label_propagation_with_history

documents = mock_documents()
seed_groups = mock_seed_groups()

graph = build_tripartite_graph(documents, seed_groups)
scores, convergence_history = label_propagation_with_history(graph, iterations=30, threshold=0.0001)
classifications = classify_reviews(graph, scores)
tf_idf_weights = calculate_tf_idf(documents)

classifications

## Balanceamento das categorias

O histograma abaixo mostra quantas reviews foram classificadas em cada categoria. Ele ajuda a verificar se o algoritmo esta concentrando tudo em uma unica categoria ou se ha distribuicao coerente entre os aspectos tecnicos.

In [ ]:
def count_classifications(classifications, seed_groups):
    counts = []
    for category, _ in seed_groups:
        total = 0
        for _, predicted_category, _, _ in classifications:
            if predicted_category == category:
                total += 1
        counts.append((category, total))
    return counts

category_counts = count_classifications(classifications, seed_groups)
category_counts

In [ ]:
categories = [item[0] for item in category_counts]
counts = [item[1] for item in category_counts]

plt.figure(figsize=(8, 4))
bars = plt.bar(categories, counts, color=["#D99058", "#5AA9C9", "#7CB342", "#8E6BBE"])
plt.title("Balanceamento de reviews classificadas por categoria")
plt.xlabel("Categoria")
plt.ylabel("Quantidade de reviews")
plt.ylim(0, max(counts) + 1)

for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width() / 2, count + 0.05, str(count), ha="center")

plt.show()

## Palavras mais importantes por categoria

As WordClouds usam os pesos TF-IDF das palavras conectadas as reviews classificadas em cada categoria. Quanto maior a palavra, maior foi seu peso acumulado dentro daquela categoria.

In [ ]:
def find_weighted_terms(tf_idf_weights, review_label):
    for current_label, weighted_terms in tf_idf_weights:
        if current_label == review_label:
            return weighted_terms
    return []


def add_weight(word_weights, word, weight):
    for index in range(len(word_weights)):
        current_word, current_weight = word_weights[index]
        if current_word == word:
            word_weights[index] = (current_word, current_weight + weight)
            return
    word_weights.append((word, weight))


def category_word_weights(classifications, tf_idf_weights, seed_groups):
    grouped = []
    for category, _ in seed_groups:
        grouped.append((category, []))

    for review_label, predicted_category, _, _ in classifications:
        terms = find_weighted_terms(tf_idf_weights, review_label)
        for index in range(len(grouped)):
            category, word_weights = grouped[index]
            if category == predicted_category:
                for word, weight in terms:
                    add_weight(word_weights, word, weight)

    return grouped


word_weights_by_category = category_word_weights(classifications, tf_idf_weights, seed_groups)
word_weights_by_category

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for axis, (category, word_weights) in zip(axes, word_weights_by_category):
    frequencies = {word: weight for word, weight in word_weights}
    axis.set_title(category)
    axis.axis("off")

    if frequencies:
        cloud = WordCloud(width=700, height=400, background_color="white", colormap="viridis")
        cloud.generate_from_frequencies(frequencies)
        axis.imshow(cloud, interpolation="bilinear")
    else:
        axis.text(0.5, 0.5, "Sem palavras", ha="center", va="center")

plt.tight_layout()
plt.show()

## Estatisticas matematicas de convergencia

A cada iteracao, o algoritmo calcula `max_delta`, ou seja, a maior mudanca absoluta entre os scores antigos e novos de todos os nos e categorias.

Quando `max_delta` fica menor que o `threshold`, a propagacao e considerada convergida.

In [ ]:
convergence_history

In [ ]:
iterations = [item[0] for item in convergence_history]
deltas = [item[1] for item in convergence_history]

convergence_summary = {
    "iterations_run": len(convergence_history),
    "initial_delta": deltas[0] if deltas else 0.0,
    "final_delta": deltas[-1] if deltas else 0.0,
    "min_delta": min(deltas) if deltas else 0.0,
    "max_delta": max(deltas) if deltas else 0.0,
    "converged_under_threshold_0_0001": bool(deltas and deltas[-1] < 0.0001),
}

convergence_summary

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(iterations, deltas, marker="o", color="#2E4057")
plt.axhline(0.0001, color="#C0392B", linestyle="--", label="threshold = 0.0001")
plt.title("Convergencia do Label Propagation")
plt.xlabel("Iteracao")
plt.ylabel("Max delta")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Conclusao

As evidencias geradas pelo notebook permitem validar tres pontos:

1. As reviews mockadas sao distribuidas entre categorias tecnicas.
2. As palavras de maior TF-IDF por categoria sao semanticamente coerentes com as seeds.
3. O Label Propagation apresenta queda de `max_delta`, demonstrando convergencia numerica ate o limiar configurado.